In [1]:
import osmnx as ox
from shapely.geometry import LineString
import pandas as pd
import ast
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

In [2]:
# Define your bounding box manually
north = 41.188771290159195
south = 41.13090363866696
west  = -8.699966476287791
east  = -8.559738123284369


bbox = (west, south, east, north)
# Fetch network from the bounding box
G_bbox = ox.graph_from_bbox(
    bbox=bbox,
    network_type="drive",   # can be 'drive', 'walk', etc.
    simplify=True,
)

# Project to local CRS (UTM Zone 29N, or Mercator 3857)
porto_crs = 3857
G = ox.project_graph(G_bbox, to_crs=porto_crs)

# Add geometry to edges that don't have it
for u, v, k, data in G.edges(keys=True, data=True):
    if 'geometry' not in data:
        point_u = (G.nodes[u]['x'], G.nodes[u]['y']) # x is long, y is lat
        point_v = (G.nodes[v]['x'], G.nodes[v]['y'])
        data['geometry'] = LineString([point_u, point_v])
        

In [3]:
# Read trajectory data of the form long, lat
traj_df = pd.read_csv("train_15.csv", index_col=0)

polylines = traj_df["POLYLINE"].to_numpy()
trajectories = []

for line in polylines:
    traj = ast.literal_eval(line)  # list of (lon, lat)
    trajectories.append(traj)


# Unclean trips, used for plotting
trips = [  gpd.GeoDataFrame(
        geometry=[Point(lon, lat) for lon, lat in traj],crs="EPSG:4326"  # convert from lat/long
        ).to_crs(porto_crs) 
    for traj in trajectories]


# Vivid colors for trips and routes
vivid_colors = [
    "#e41a1c",  # red
    "#377eb8",  # blue
    "#4daf4a",  # green
    "#984ea3",  # purple
    "#ff7f00",  # orange
    "#ffff33",  # yellow
    "#a65628",  # brown
    "#f781bf",  # pink
    "#999999",  # gray
    "#66c2a5",  # teal
    "#fc8d62",  # salmon
    "#8da0cb",  # light blue
    "#000000",  # black
    "#a6d854",  # lime
    "#ffd92f"   # gold
]

## Unclean Trips

### Plotting unclean GPS Trajectory data in this section

In [4]:
# Base graph plot
fig, ax = ox.plot_graph(
    G,
    figsize=(15,9),
    bgcolor="white",
    node_size=6,
    node_color="lightgray",
    node_zorder=1,
    edge_color="lightgray",
    edge_linewidth=0.8,
    show=False,
    close=False,
)

# Plot each trajectory
for index, traj in enumerate(trips):
    x, y = traj.geometry.x, traj.geometry.y
    color = vivid_colors[index % len(vivid_colors)]  # wrap around if >15 trips
    ax.plot(
        x, y,
        color=color,
        marker="o",
        markersize=4,
        markeredgecolor="black",
        markeredgewidth=0.3,
        linewidth=1.5,
        alpha=0.9,
        zorder=3,
        label=f"Trip {index+1}"
    )

# Legend and title
ax.legend(
    loc="upper right",
    fontsize=8,
    frameon=True,
    facecolor="white",
    edgecolor="black"
)

ax.set_title("First 15 Porto Trips on Road Network", fontsize=14, y=0.95)
fig.savefig("LCSS_Images/unclean_gps_plot_part_2.png", dpi=300, bbox_inches="tight", pad_inches=0)

plt.close(fig)
